In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import wikipedia

print('✅ Imports successful!')


In [ ]:
# ---- OpenAI ----
os.environ['OPENAI_API_KEY'] = 'sk-...'     # Required for embeddings

# ---- Pinecone ----
PINECONE_API_KEY    = 'pc-...'              # https://app.pinecone.io
PINECONE_INDEX_NAME = 'rag-showdown'
PINECONE_CLOUD      = 'aws'                 # or 'gcp' / 'azure'
PINECONE_REGION     = 'us-east-1'

# ---- Azure AI Search ----
AZURE_SEARCH_ENDPOINT   = 'https://.search.windows.net'
AZURE_SEARCH_API_KEY    = ''
AZURE_SEARCH_INDEX_NAME = 'rag-showdown'

# ---- Cohere (extension only) ----
COHERE_API_KEY = ''  # Leave blank to skip extension

# ---- Shared ----
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM   = 1536
TOP_K           = 5

print('✅ Configuration ready!')

In [ ]:
SEED_TOPICS = [
    'Artificial intelligence', 'Machine learning', 'Deep learning', 'Neural network',
    'Natural language processing', 'Computer vision', 'Quantum computing',
    'CRISPR', 'Gene therapy', 'Protein folding',
    'World War II', 'French Revolution', 'Roman Empire', 'Industrial Revolution',
    'Cold War', 'Space Race', 'Manhattan Project',
    'Amazon rainforest', 'Himalayas', 'Sahara Desert', 'Great Barrier Reef',
    'Renaissance', 'Jazz music', 'Ballet', 'Olympic Games', 'Nobel Prize',
    'Diabetes', 'Cancer', 'Alzheimer\'s disease', 'COVID-19', 'Vaccine', 'Antibiotics',
    'Stock market', 'Bitcoin', 'Inflation', 'Gross domestic product',
    'International Monetary Fund', 'Supply chain',
]

wikipedia.set_lang('en')
raw_articles = []
for topic in tqdm(SEED_TOPICS, desc='Fetching Wikipedia'):
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        raw_articles.append({'title': page.title, 'content': page.content, 'url': page.url})
    except Exception as e:
        print(f'  ⚠️  Skipping "{topic}": {e}')

print(f'\n✅ Fetched {len(raw_articles)} articles')

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', '']
)

all_docs: list[Document] = []
for art in raw_articles:
    chunks = splitter.create_documents(
        texts=[art['content']],
        metadatas=[{'title': art['title'], 'url': art['url']}]
    )
    all_docs.extend(chunks)

all_docs = all_docs[:500]  # Cap at 500 per spec
texts = [d.page_content for d in all_docs]
metas = [d.metadata      for d in all_docs]

print(f'✅ Total chunks: {len(all_docs)}')
print(f'   Sample ({len(all_docs[0].page_content)} chars): {all_docs[0].page_content[:200]}...')

In [ ]:
embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)

print(f'Embedding {len(all_docs)} chunks with "{EMBEDDING_MODEL}"...')
t0 = time.perf_counter()
all_vectors = embedder.embed_documents(texts)
elapsed = time.perf_counter() - t0

print(f'✅ Embedded {len(all_vectors)} chunks in {elapsed:.1f}s  |  dim={len(all_vectors[0])}')

In [ ]:

import faiss

t0 = time.perf_counter()
faiss_raw = faiss.IndexFlatL2(EMBEDDING_DIM)
faiss_raw.add(np.array(all_vectors, dtype=np.float32))
build_ms = (time.perf_counter() - t0) * 1000

print(f'✅ FAISS IndexFlatL2 built in {build_ms:.1f}ms  |  {faiss_raw.ntotal} vectors')

In [ ]:

faiss_store     = FAISS.from_documents(all_docs, embedder)
faiss_retriever = faiss_store.as_retriever(search_kwargs={'k': TOP_K})
print('✅ LangChain FAISS vectorstore ready')

In [ ]:
     

faiss_store     = FAISS.from_documents(all_docs, embedder)
faiss_retriever = faiss_store.as_retriever(search_kwargs={'k': TOP_K})
print('✅ LangChain FAISS vectorstore ready')
     

BENCHMARK_QUERIES = [
    'What is the role of transformers in NLP?',
    'Explain the causes of World War II',
    'How does CRISPR gene editing work?',
    'What are the effects of climate change on the Amazon?',
    'Describe the Bitcoin mining process',
    'What is the mechanism of mRNA vaccines?',
    'How did the Cold War end?',
    'Explain supply chain disruption causes',
    'What caused the French Revolution?',
    'How does deep learning differ from machine learning?',
    'What are the symptoms of Alzheimers disease?',
    'What is inflation and how is it measured?',
    'What is the history of the Olympic Games?',
    'Explain quantum entanglement',
    'How does the stock market work?',
    'What are the causes of cancer?',
    'Describe the Renaissance period in Europe',
    'How are antibiotics made?',
    'What is gross domestic product?',
    'How do neural networks learn?',
]

faiss_latencies = []
for q in tqdm(BENCHMARK_QUERIES, desc='FAISS benchmark'):
    t0 = time.perf_counter()
    faiss_retriever.invoke(q)
    faiss_latencies.append((time.perf_counter() - t0) * 1000)

faiss_p50 = np.percentile(faiss_latencies, 50)
faiss_p95 = np.percentile(faiss_latencies, 95)
print(f'\n📊 FAISS  p50={faiss_p50:.1f}ms  p95={faiss_p95:.1f}ms')

# Sample result
for i, r in enumerate(faiss_retriever.invoke(BENCHMARK_QUERIES[0])):
    print(f'  [{i+1}] [{r.metadata["title"]}] {r.page_content[:100]}...')

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    print(f'Creating index "{PINECONE_INDEX_NAME}"...')
    pc.create_index(
        name=PINECONE_INDEX_NAME, dimension=EMBEDDING_DIM, metric='cosine',
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status['ready']:
        print('  Waiting...'); time.sleep(5)

pinecone_index = pc.Index(PINECONE_INDEX_NAME)
print('✅ Pinecone index ready:', pinecone_index.describe_index_stats())

In [ ]:
BATCH = 100
print(f'Upserting {len(all_docs)} vectors to Pinecone...')
t0 = time.perf_counter()

for i in tqdm(range(0, len(all_docs), BATCH), desc='Upserting'):
    pinecone_index.upsert(vectors=[
        {
            'id': f'doc-{i+j}',
            'values': all_vectors[i+j],
            'metadata': {**metas[i+j], 'text': texts[i+j][:500]}
        }
        for j in range(min(BATCH, len(all_docs)-i))
    ])

print(f'✅ Upserted in {time.perf_counter()-t0:.1f}s')
print('   Stats:', pinecone_index.describe_index_stats())

In [ ]:
pinecone_store     = PineconeVectorStore(index=pinecone_index, embedding=embedder, text_key='text')
pinecone_retriever = pinecone_store.as_retriever(search_kwargs={'k': TOP_K})

print('🔍 Semantic search demo:')
for i, r in enumerate(pinecone_retriever.invoke(BENCHMARK_QUERIES[0])):
    print(f'  [{i+1}] [{r.metadata.get("title","N/A")}] {r.page_content[:100]}...')
     

In [ ]:
# Metadata filter demo
print('🔍 Search with metadata filter (title=Artificial intelligence):')
filtered = pinecone_store.similarity_search(
    'What are neural networks?', k=3,
    filter={'title': {'$eq': 'Artificial intelligence'}}
)
for i, r in enumerate(filtered):
    print(f'  [{i+1}] [{r.metadata.get("title","N/A")}] {r.page_content[:100]}...')

In [ ]:

pinecone_latencies = []
for q in tqdm(BENCHMARK_QUERIES, desc='Pinecone benchmark'):
    t0 = time.perf_counter()
    pinecone_retriever.invoke(q)
    pinecone_latencies.append((time.perf_counter() - t0) * 1000)

pinecone_p50 = np.percentile(pinecone_latencies, 50)
pinecone_p95 = np.percentile(pinecone_latencies, 95)
print(f'📊 Pinecone  p50={pinecone_p50:.1f}ms  p95={pinecone_p95:.1f}ms')

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SearchField, SemanticConfiguration, SemanticSearch,
    SemanticPrioritizedFields, SemanticField
)
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential

cred         = AzureKeyCredential(AZURE_SEARCH_API_KEY)
idx_client   = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=cred)
srch_client  = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT,
                             index_name=AZURE_SEARCH_INDEX_NAME, credential=cred)
print('✅ Azure AI Search clients initialized')

In [ ]:
fields = [
    SimpleField(name='id',       type=SearchFieldDataType.String, key=True),
    SearchableField(name='content', type=SearchFieldDataType.String, analyzer_name='en.microsoft'),
    SimpleField(name='title',    type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='url',      type=SearchFieldDataType.String),
    SearchField(
        name='embedding',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIM,
        vector_search_profile_name='hnsw-profile'
    ),
]

index_def = SearchIndex(
    name=AZURE_SEARCH_INDEX_NAME,
    fields=fields,
    vector_search=VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name='hnsw-algo')],
        profiles=[VectorSearchProfile(name='hnsw-profile', algorithm_configuration_name='hnsw-algo')]
    ),
    semantic_search=SemanticSearch(configurations=[
        SemanticConfiguration(
            name='semantic-cfg',
            prioritized_fields=SemanticPrioritizedFields(
                content_fields=[SemanticField(field_name='content')],
                keywords_fields=[SemanticField(field_name='title')]
            )
        )
    ])
)

result = idx_client.create_or_update_index(index_def)
print(f'✅ Azure index "{result.name}" ready (hybrid BM25 + vector)')

In [ ]:
BATCH = 100
print(f'Uploading {len(all_docs)} docs to Azure AI Search...')
t0 = time.perf_counter()

for i in tqdm(range(0, len(all_docs), BATCH), desc='Azure upload'):
    batch = [
        {'id': f'doc-{i+j}', 'content': texts[i+j],
         'title': metas[i+j].get('title',''), 'url': metas[i+j].get('url',''),
         'embedding': all_vectors[i+j]}
        for j in range(min(BATCH, len(all_docs)-i))
    ]
    srch_client.upload_documents(documents=batch)

print(f'✅ Uploaded in {time.perf_counter()-t0:.1f}s')

In [ ]:
def azure_hybrid_search(query: str, top_k: int = TOP_K):
    """BM25 keyword + vector hybrid search."""
    qvec = embedder.embed_query(query)
    results = srch_client.search(
        search_text=query,
        vector_queries=[VectorizedQuery(vector=qvec, k_nearest_neighbors=top_k, fields='embedding')],
        select=['id', 'content', 'title', 'url'],
        top=top_k
    )
    return list(results)

print('🔍 Hybrid search demo:')
for i, r in enumerate(azure_hybrid_search(BENCHMARK_QUERIES[0])):
    print(f'  [{i+1}] score={r.get("@search.score", 0):.4f} [{r["title"]}] {r["content"][:100]}...')

In [ ]:
azure_latencies = []
for q in tqdm(BENCHMARK_QUERIES, desc='Azure benchmark'):
    t0 = time.perf_counter()
    azure_hybrid_search(q)
    azure_latencies.append((time.perf_counter() - t0) * 1000)

azure_p50 = np.percentile(azure_latencies, 50)
azure_p95 = np.percentile(azure_latencies, 95)
print(f'📊 Azure AI Search  p50={azure_p50:.1f}ms  p95={azure_p95:.1f}ms')

In [ ]:

summary = pd.DataFrame({
    'Vector DB':   ['FAISS (in-memory)', 'Pinecone Serverless', 'Azure AI Search (Hybrid)'],
    'p50 (ms)':    [round(faiss_p50,1),  round(pinecone_p50,1),  round(azure_p50,1)],
    'p95 (ms)':    [round(faiss_p95,1),  round(pinecone_p95,1),  round(azure_p95,1)],
    'Search Type': ['Exact k-NN (L2)',   'Approx cosine',         'BM25 + Vector'],
    'Cost':        ['Free',              '$$',                    '$$$'],
})

print('=' * 75)
print('📊  LATENCY BENCHMARK  (20 queries, embedding time included)')
print('=' * 75)
print(summary.to_string(index=False))
print('=' * 75)

In [ ]:
dbs    = ['FAISS', 'Pinecone', 'Azure AI']
colors = ['#4C72B0', '#DD8452', '#55A868']
p50s   = [faiss_p50, pinecone_p50, azure_p50]
p95s   = [faiss_p95, pinecone_p95, azure_p95]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Vector DB Latency Showdown — FAISS vs Pinecone vs Azure AI Search',
             fontsize=13, fontweight='bold')

for ax, vals, label in [(ax1, p50s, 'p50 Latency (ms)'), (ax2, p95s, 'p95 Latency (ms)')]:
    bars = ax.bar(dbs, vals, color=colors, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('ms')
    ax.set_ylim(0, max(vals) * 1.4)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
                f'{v:.0f}ms', ha='center', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('latency_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved latency_benchmark.png')

In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))

bp = ax.boxplot(
    [faiss_latencies, pinecone_latencies, azure_latencies],
    labels=['FAISS', 'Pinecone', 'Azure AI Search'],
    patch_artist=True, notch=True,
    medianprops=dict(color='white', linewidth=2)
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)

ax.set_title('Latency Distribution (20 queries)', fontweight='bold', fontsize=13)
ax.set_ylabel('Latency (ms)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('latency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if not COHERE_API_KEY:
    print('⚠️  COHERE_API_KEY not set — skipping extension.')
else:
    import cohere
    co = cohere.Client(COHERE_API_KEY)

    def retrieve_and_rerank(query, top_k_retrieve=10, top_k_final=5):
        docs = faiss_store.similarity_search(query, k=top_k_retrieve)
        passages = [d.page_content for d in docs]
        reranked = co.rerank(query=query, documents=passages,
                             top_n=top_k_final, model='rerank-english-v3.0')
        return [(docs[r.index], r.relevance_score) for r in reranked.results]

    def mrr(ranked_docs, relevant_title):
        for rank, (doc, _) in enumerate(ranked_docs, 1):
            if relevant_title.lower() in doc.metadata.get('title','').lower():
                return 1.0 / rank
        return 0.0

    eval_set = [
        ('How does backpropagation work?',      'Deep learning'),
        ('What causes inflation?',              'Inflation'),
        ('How are mRNA vaccines made?',         'Vaccine'),
        ('What are symptoms of Alzheimers?',    "Alzheimer's disease"),
        ('How did WWII start?',                 'World War II'),
    ]

    before, after = [], []
    for query, rel in eval_set:
        before_docs = [(d, 0) for d in faiss_store.similarity_search(query, k=5)]
        before.append(mrr(before_docs, rel))
        after.append(mrr(retrieve_and_rerank(query), rel))

    print(f'\n🏅 MRR@5 Before re-rank: {np.mean(before):.3f}')
    print(f'🏅 MRR@5 After  re-rank: {np.mean(after):.3f}')
    print(f'   Improvement: +{(np.mean(after)-np.mean(before))*100:.1f}pp')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Before Re-rank', 'After Re-rank'],
           [np.mean(before), np.mean(after)],
           color=['#DD8452', '#55A868'], edgecolor='white')
    ax.set_title('MRR@5: FAISS → Cohere Re-rank', fontweight='bold')
    ax.set_ylabel('Mean Reciprocal Rank'); ax.set_ylim(0, 1.1)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('mrr_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
     